# Задание Ultra Pro

Решите задачу распознавания 20 русских писателей. Это подразумевает и больший размер базы для обучения соответственно. Ячейка для скачивания базы уже включена в ноутбук задания.


 В задании необходимо выполнить следующие пункты:

  1. Загрузить саму базу по ссылке и подговить файлы базы для обработки.
  2. Создать обучающую и проверочную выборки, обратив особое внимание на балансировку базы: количество примеров каждого класса должно быть примерно одного порядка. При этом для разбивки необходимо применить цикл. Проверочная выборка должна быть 20% от общей выборки.
  3. Подготовьте выборки для обучения и обучите сеть. Добейтесь результата точности сети не менее 95% на проверочной выборке модели Bag of Words и 75-80% - для модели Embedding.
   


## Загрузка данных

In [1]:
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip', None, quiet=True)

!unzip -qo 20writers.zip -d writers/
!ls writers

Беляев.txt    Гончаров.txt     Каверин.txt    Лесков.txt     Толстой.txt
Булгаков.txt  Горький.txt      Катаев.txt     Носов.txt      Тургенев.txt
Васильев.txt  Грибоедов.txt    Куприн.txt     Пастернак.txt  Чехов.txt
Гоголь.txt    Достоевский.txt  Лермонтов.txt  Пушкин.txt     Шолохов.txt


In [2]:
import os
import zipfile
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from tensorflow.keras import utils
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
base_path = 'writers' # задание пути к папке с текстами

# создание списков
train_texts = []
test_texts = []
class_names = []

txt_paths = []

# поиск всех txt-файлов внутри папок
for root, dirs, files in os.walk(base_path):
    for file_name in files:
        if file_name.endswith('.txt'):
            txt_paths.append(os.path.join(root, file_name))

print('Найдено txt-файлов:', len(txt_paths))

# чтение каждого текстового файла
for file_path in txt_paths:
    # получение имени автора из названия файла
    author = os.path.basename(file_path).replace('.txt', '')
    class_names.append(author)
    with open(file_path, 'r', encoding='utf-8') as f: # чтение текста
        text = f.read()

    split = int(len(text) * 0.8)

    # разделение выборок
    train_texts.append(text[:split])
    test_texts.append(text[split:])

num_classes = len(class_names)

print('Количество классов:', num_classes)
print(class_names)

Найдено txt-файлов: 20
Количество классов: 20
['Толстой', 'Беляев', 'Булгаков', 'Васильев', 'Катаев', 'Пушкин', 'Грибоедов', 'Тургенев', 'Куприн', 'Лермонтов', 'Гоголь', 'Шолохов', 'Каверин', 'Горький', 'Гончаров', 'Достоевский', 'Пастернак', 'Чехов', 'Носов', 'Лесков']


In [4]:
# задание размеров параметров
VOCAB_SIZE = 5000
WIN_SIZE = 1000
WIN_HOP = 100

# создание токенов
tokenizer = Tokenizer(num_words=VOCAB_SIZE, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n\xa0', lower=True, split=' ', oov_token='unknown')

# обучение токенов
tokenizer.fit_on_texts(train_texts)

# преобразование текстов в числовые последовательности
train_seq = tokenizer.texts_to_sequences(train_texts)
test_seq = tokenizer.texts_to_sequences(test_texts)

# класс обработки текстовых последовательностей
class DataProcessor:
    def __init__(self, win_size, win_hop):
        self.win_size = win_size
        self.win_hop = win_hop

    # нарезка последовательности на окна
    def split_sequence(self, sequence):
        windows = []

        # формирование окон заданного размера
        for i in range(0, len(sequence) - self.win_size + 1, self.win_hop):
            windows.append(sequence[i:i + self.win_size])

        return windows

    # формирование обучающей и проверочной выборки
    def make_train_test(self, seq_list, max_train=None, max_test=None):
        x_train = []
        y_train = []
        x_test = []
        y_test = []

        # обработка каждого класса
        for class_index in range(len(seq_list)):
            # нарезка текста класса на окна
            windows = self.split_sequence(seq_list[class_index])

            # перемешивание окон
            np.random.shuffle(windows)

            # вычисление границы разделения на 80% и 20%
            split = int(len(windows) * 0.8)

            train_windows = windows[:split]
            test_windows = windows[split:]

            # добавление меток окнам
            x_train += train_windows
            y_train += [utils.to_categorical(class_index, len(seq_list))] * len(train_windows)

            x_test += test_windows
            y_test += [utils.to_categorical(class_index, len(seq_list))] * len(test_windows)

        return np.array(x_train), np.array(y_train), np.array(x_test), np.array(y_test)

In [5]:
np.random.seed(42)

# создание общего списка последовательностей
all_seq = []

# объединение обучающей и проверочной части каждого автора
for i in range(num_classes):
    all_seq.append(train_seq[i] + test_seq[i])

processor = DataProcessor(WIN_SIZE, WIN_HOP)

# формирование обучающей и проверочной выборки
x_train_seq, y_train, x_test_seq, y_test = processor.make_train_test(all_seq, max_train=300, max_test=75)

print('x_train_seq:', x_train_seq.shape)
print('y_train:', y_train.shape)
print('x_test_seq:', x_test_seq.shape)
print('y_test:', y_test.shape)

x_train_seq: (71792, 1000)
y_train: (71792, 20)
x_test_seq: (17958, 1000)
y_test: (17958, 20)


In [ ]:
# преобразование последовательностей
x_train_bow = tokenizer.sequences_to_matrix(x_train_seq.tolist(), mode='tfidf').astype('float32')
x_test_bow = tokenizer.sequences_to_matrix(x_test_seq.tolist(), mode='tfidf').astype('float32')

print(x_train_bow.shape)
print(x_test_bow.shape)

model_bow = Sequential()

model_bow.add(Dense(128, input_dim=VOCAB_SIZE, activation='relu'))
model_bow.add(Dropout(0.3))
model_bow.add(Dense(num_classes, activation='softmax'))

model_bow.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history_bow = model_bow.fit(x_train_bow, y_train, epochs=8, batch_size=32, validation_data=(x_test_bow, y_test), callbacks=[early_stop])

plt.figure(figsize=(5, 3))
plt.plot(history_bow.history['accuracy'], label='Обучение')
plt.plot(history_bow.history['val_accuracy'], label='Проверка')
plt.title('Bag of Words — точность')
plt.xlabel('Эпоха')
plt.ylabel('Точность')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot(history_bow.history['loss'], label='Обучение')
plt.plot(history_bow.history['val_loss'], label='Проверка')
plt.title('Bag of Words — ошибка')
plt.xlabel('Эпоха')
plt.ylabel('Ошибка')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# создание модели
model_emb = Sequential()

model_emb.add(Embedding(VOCAB_SIZE, 64, input_length=WIN_SIZE))
model_emb.add(Conv1D(128, 5, activation='relu'))
model_emb.add(GlobalMaxPooling1D())
model_emb.add(Dense(100, activation='relu'))
model_emb.add(Dense(num_classes, activation='softmax'))

model_emb.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_emb = model_emb.fit(x_train_seq, y_train, epochs=10, batch_size=32, validation_data=(x_test_seq, y_test))

plt.figure(figsize=(5, 3))
plt.plot(history_emb.history['accuracy'], label='Обучение')
plt.plot(history_emb.history['val_accuracy'], label='Проверка')
plt.title('Embedding — точность')
plt.xlabel('Эпоха')
plt.ylabel('Точность')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot(history_emb.history['loss'], label='Обучение')
plt.plot(history_emb.history['val_loss'], label='Проверка')
plt.title('Embedding — ошибка')
plt.xlabel('Эпоха')
plt.ylabel('Ошибка')
plt.legend()
plt.grid()
plt.show()